# KAN Verification Functions

In [1]:
import numpy as np
from src.custom_fastkan import FastKAN, FastKANLayer
import gurobipy as gp
from gurobipy import GRB
import torch
from torch import nn
import time
import torch.optim as optim

def fit_line_through_points(x1, y1, x2, y2):
    slope = (y2 - y1) / (x2 - x1)
    intercept = y1 - slope * x1
    return slope, intercept

def find_all_bspline_segments(layer, input_index, output_index, max_segments):
    n_samples = 25
    # Fetch curve data ONCE
    x_tensor, y_tensor = layer.plot_curve(input_index, output_index, num_pts=n_samples)
    x_points, y_points = x_tensor.detach().cpu().numpy(), y_tensor.detach().cpu().numpy()
    
    # 1. Precompute errors for all possible segments (ONCE)
    errors = np.full((n_samples, n_samples), np.inf)
    for start_idx in range(n_samples-1):
        x_start = x_points[start_idx]
        y_start = y_points[start_idx]
        for end_idx in range(start_idx + 1, n_samples):
            x_end = x_points[end_idx]
            y_end = y_points[end_idx]
            slope, intercept = fit_line_through_points(x_start, y_start, x_end, y_end)
            segment_x = x_points[start_idx:end_idx+1]
            segment_y = y_points[start_idx:end_idx+1]
            predicted_y = slope * segment_x + intercept
            segment_error = np.max(np.abs(predicted_y - segment_y))
            errors[start_idx, end_idx] = segment_error
    
    # 2. Dynamic programming to find optimal segmentation (ONCE)
    dp_table = np.full((max_segments, n_samples), np.inf)
    backtrack = np.zeros((max_segments, n_samples), dtype=int)
    
    # Base Case: 1 segment
    for j in range(1, n_samples):
        dp_table[0, j] = errors[0, j]
    
    # Recurrence
    for i in range(1, max_segments):
        for j in range(i+1, n_samples):
            # We try breaking at k, where k is between the start of this segment number and current point
            # k is the END of the previous segment
            for k in range(i, j): 
                if dp_table[i-1, k] == np.inf: continue
                if errors[k, j] == np.inf: continue
                
                curr_error = max(dp_table[i-1, k], errors[k, j])
                if curr_error < dp_table[i, j]:
                    dp_table[i, j] = curr_error
                    backtrack[i, j] = k

    # 3. Extract results for EVERY segment count from 1 to max_segments
    results = {}
    
    for seg_count in range(1, max_segments + 1):
        row_idx = seg_count - 1
        final_error = dp_table[row_idx, n_samples-1]
        
        # If optimization failed for this count, skip or return infinity
        if final_error == np.inf:
            results[seg_count] = ([], np.inf)
            continue

        # Reconstruct segments for this specific count using the backtrack table
        segments = []
        curr_seg_end = n_samples - 1
        
        for i in range(row_idx, -1, -1):
            if i > 0:
                prev_seg_end = backtrack[i, curr_seg_end]
            else:
                prev_seg_end = 0
            
            x1, y1 = x_points[prev_seg_end], y_points[prev_seg_end]
            x2, y2 = x_points[curr_seg_end], y_points[curr_seg_end]
            slope, intercept = fit_line_through_points(x1, y1, x2, y2)
            segments.insert(0, (x1, x2, slope, intercept))
            
            curr_seg_end = prev_seg_end
        
        results[seg_count] = (segments, final_error)
        
    return results

def calculate_bspline_lipschitz_constant(layer: 'FastKANLayer', input_idx: int, output_idx: int, num_pts: int = 1000, min_x: float = None, max_x: float = None) -> float:
    if min_x is None:
        min_x = layer.rbf.grid_min
    if max_x is None:
        max_x = layer.rbf.grid_max
    x_tensor, y_tensor = layer.plot_curve(input_idx, output_idx, num_pts=num_pts)
    x_np = x_tensor.detach().cpu().numpy()
    y_np = y_tensor.detach().cpu().numpy()
    mask = (x_np >= min_x) & (x_np <= max_x)
    x_np = x_np[mask]
    y_np = y_np[mask]

    dx = np.diff(x_np)
    dy = np.diff(y_np)
    nonzero_dx = dx != 0
    slopes = np.zeros_like(dx)
    slopes[nonzero_dx] = dy[nonzero_dx] / dx[nonzero_dx]
    lipschitz_constant = np.max(np.abs(slopes))
    return lipschitz_constant

def compute_dp_tables_lipschitz(kan_model, max_segments=15):
    error_tables = {}
    segments_tables = {}
    lipschitz_constants = {}
    
    for layer_idx, layer in enumerate(kan_model.layers):
        input_dim = layer.input_dim
        output_dim = layer.output_dim
        print(f"Analyzing Layer {layer_idx}: {input_dim} inputs -> {output_dim} outputs")
        
        for input_idx in range(input_dim):
            for output_idx in range(output_dim):
                spline_key = (layer_idx, input_idx, output_idx)
                all_results = find_all_bspline_segments(layer, input_idx, output_index=output_idx, max_segments=max_segments)
                
                error_table = {}
                segments_table = {}
                
                for k, (segs, err) in all_results.items():
                    segments_table[k] = segs
                    error_table[k] = err
                lipschitz_constant = calculate_bspline_lipschitz_constant(layer=layer, input_idx=input_idx, output_idx=output_idx)
                
                error_tables[spline_key] = error_table
                segments_tables[spline_key] = segments_table
                lipschitz_constants[spline_key] = lipschitz_constant
    
    return error_tables, segments_tables, lipschitz_constants


def weight_dp_tables_lipschitz(kan_model, error_tables, segments_tables, lipschitz_constants):
    node_sensitivities = {}
    num_layers = len(kan_model.layers)
    
    # Intialize final layer with all 1's
    final_layer = kan_model.layers[num_layers - 1]
    for out_idx in range(final_layer.output_dim):
        node_sensitivities[(num_layers, out_idx)] = 1.0

    # Calculate sensitivity for the inputs of a layer based on the output layer's sensitivities
    layer_idx = num_layers - 1
    while layer_idx >= 0:
        current_layer = kan_model.layers[layer_idx]
        for input_idx in range(current_layer.input_dim):
            sensitivity_sum = 0.0
            for output_idx in range(current_layer.output_dim):
                lipschitz_constant = lipschitz_constants.get((layer_idx, input_idx, output_idx), 0.0)
                child_node_sensitivity = node_sensitivities[(layer_idx + 1, output_idx)]
                sensitivity_sum += lipschitz_constant * child_node_sensitivity
            node_sensitivities[(layer_idx, input_idx)] = sensitivity_sum
        layer_idx -= 1

    # Weight the error table (just do sensitivities * table)
    weighted_error_tables = {}
    for layer_idx, layer in enumerate(kan_model.layers):
        for input_idx in range(layer.input_dim):
            for output_idx in range(layer.output_dim):
                current_bspline_key = (layer_idx, input_idx, output_idx)
                # The error of a spline is added directly to its destination node (output_idx)
                sensitivity_of_current_output_node = node_sensitivities[(layer_idx + 1, output_idx)]
                weighted_table_for_bspline = {}
                for pair in error_tables[current_bspline_key].items():
                    num_segments = pair[0]
                    error = pair[1]
                    weighted_table_for_bspline[num_segments] = error * sensitivity_of_current_output_node
                weighted_error_tables[current_bspline_key] = weighted_table_for_bspline
    return weighted_error_tables


def solve_best_segment_allocation(weighted_error_tables, target_max_error):
    model = gp.Model()
    model.setParam("OutputFlag", 0)
    # Create binary variable x[spline_key, num_segments] for every choice and every spline
    x = {}
    binary_vars_for_splines = {} 
    for spline_key, num_segment_options in weighted_error_tables.items():
        binary_vars_for_splines[spline_key] = []
        for pair in num_segment_options.items():
            k_segments = pair[0]
            error_val = pair[1]
            if np.isinf(error_val):
                continue
            x[(spline_key, k_segments)] = model.addVar(vtype=GRB.BINARY, obj=k_segments)
            binary_vars_for_splines[spline_key].append(x[(spline_key, k_segments)])
    model.update()

    # Make sure we can only pick 1 variable
    for pair in binary_vars_for_splines.items():
        spline_key = pair[0]
        variables = pair[1]
        model.addConstr(gp.quicksum(variables) == 1)
    # Make sure we're under the max error
    total_error_expr = gp.quicksum(weighted_error_tables[s_key][k] * x[(s_key, k)] for s_key, k in x.keys())
    model.addConstr(total_error_expr <= target_max_error)

    # Minimize the total number of segments
    total_segments_expr = gp.quicksum(
        k * x[(s_key, k)] 
        for s_key, k in x.keys()
    )
    model.setObjective(total_segments_expr, GRB.MINIMIZE)

    # Optimize!
    model.optimize()
    optimal_allocation = {}
    if model.status == GRB.OPTIMAL:
        for (spline_key, k_segments), variable in x.items():
            if variable.X > 0.5:
                optimal_allocation[spline_key] = k_segments
        print(f"Optimal allocation found with {model.objVal} segments.")
        return optimal_allocation, model.objVal
    elif model.status == GRB.INFEASIBLE:
        return None, float('inf')
    else:
        return None, float('inf')

def build_kan_milp_model(kan_shape, segments_tables, error_tables, optimal_allocation, x_min_vec, x_max_vec):
    model = gp.Model()
    model.setParam("OutputFlag", 0)
    input_dim = kan_shape[0] 
    all_layer_variables = []

    # Create Input Variables
    current_layer_range_variables = []
    for i in range(input_dim):
        input_range_variable = model.addVar(lb=x_min_vec[i], ub=x_max_vec[i])
        current_layer_range_variables.append(input_range_variable)
    all_layer_variables.append(current_layer_range_variables)

    # Build Hidden Layers Sequentially
    num_transitions = len(kan_shape) - 1 # If shape is [784, 32, 10], we have 2 transitions: 0->1 and 1->2
    for layer_idx in range(num_transitions):
        layer_input_dimension = kan_shape[layer_idx]
        layer_output_dimension = kan_shape[layer_idx + 1]
        
        # Initialize output variables for this layer (by creating our list of inputs to the next layer)
        next_layer_input_ranges = []
        for i in range(layer_output_dimension): 
            next_layer_input_ranges.append(gp.LinExpr()) # ex. layer2_neuron_i = layer1_neuron1 + layer1_neuron2 + ... (linear expression!)
        
        for src_idx in range(layer_input_dimension):
            for dst_idx in range(layer_output_dimension):
                # Extract PWL Points
                num_segs = optimal_allocation[(layer_idx, src_idx, dst_idx)]
                seg_data = segments_tables[(layer_idx, src_idx, dst_idx)][num_segs] # table has all possible segment allocations, only extract the optimal one (num_segs)
                x_pts, y_pts = [], []
                for idx, (x1, x2, slope, intercept) in enumerate(seg_data):
                    x_pts.append(x1)
                    y_pts.append(slope * x1 + intercept)
                    if idx == len(seg_data) - 1: # Add end point of last segment
                        x_pts.append(x2)
                        y_pts.append(slope * x2 + intercept)

                # Create output variable for each bspline (adding in error)
                approx_error = error_tables[(layer_idx, src_idx, dst_idx)][num_segs]
                error_var = model.addVar(lb=-approx_error, ub=approx_error)
                src_var = current_layer_range_variables[src_idx]
                result_var = model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY)
                model.addGenConstrPWL(src_var, result_var, x_pts, y_pts) # "The relationship between src_var and result_var must follow the path connected by the dots"
                next_layer_input_ranges[dst_idx] += result_var + error_var # fill in the output linear expressions we defined earlier (for each neuron in the next layer)

        # Create variables for next layer nodes
        next_layer_range_variables = []
        for j in range(layer_output_dimension):
            next_layer_range_variable = model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY)
            model.addConstr(next_layer_range_variable == next_layer_input_ranges[j])
            next_layer_range_variables.append(next_layer_range_variable)
        current_layer_range_variables = next_layer_range_variables
        
        all_layer_variables.append(current_layer_range_variables)

    model.update()
    return model, all_layer_variables

def get_spline_bounds(segments, x_min, x_max):
    y_min, y_max = np.inf, -np.inf
    for (sx1, sx2, slope, intercept) in segments:
        # Find the intersection of the segment domain and input domain
        overlap_start = max(sx1, x_min)
        overlap_end = min(sx2, x_max)
        # If they overlap, evaluate the line at the edges
        if overlap_start <= overlap_end:
            val_start = slope * overlap_start + intercept
            val_end = slope * overlap_end + intercept
            y_min = min(y_min, val_start, val_end)
            y_max = max(y_max, val_start, val_end)
    return y_min, y_max


def propagate_kan_intervals(kan_shape, segments_tables, error_tables, optimal_allocation, input_lb, input_ub):
    layer_bounds = []
    # Input layer
    current_lb = input_lb
    current_ub = input_ub
    layer_bounds.append((current_lb, current_ub))
    num_transitions = len(kan_shape) - 1

    # Propagate across all layers 
    for layer_idx in range(num_transitions):
        in_dim = kan_shape[layer_idx]
        out_dim = kan_shape[layer_idx + 1]
        next_lb = np.zeros(out_dim)
        next_ub = np.zeros(out_dim)
        for dst in range(out_dim):
            # Min_sum = Sum(Min_parts), Max_sum = Sum(Max_parts)
            total_min = 0.0
            total_max = 0.0
            for src in range(in_dim):
                x_min = current_lb[src]
                x_max = current_ub[src]
                num_segs = optimal_allocation[(layer_idx, src, dst)]
                segs = segments_tables[(layer_idx, src, dst)][num_segs]

                approx_error = error_tables[(layer_idx, src, dst)][num_segs]

                s_min, s_max = get_spline_bounds(segs, x_min, x_max)
                total_min += s_min - approx_error
                total_max += s_max + approx_error
            next_lb[dst] = total_min
            next_ub[dst] = total_max
        layer_bounds.append((next_lb, next_ub))
        current_lb, current_ub = next_lb, next_ub
    return layer_bounds

def solve_kan_interval_milp(kan_shape, segments_tables, error_tables, optimal_allocation, 
                            output_layer_mip_gap, x_min_vec, x_max_vec, time_limit=60):
    
    # 1. Precompute Bounds (Interval Propagation)
    precomputed_bounds = propagate_kan_intervals(
        kan_shape, 
        segments_tables, 
        error_tables,
        optimal_allocation, 
        x_min_vec, 
        x_max_vec
    )
    
    # 2. Build Model
    model, all_layer_variables = build_kan_milp_model(
        kan_shape, 
        segments_tables, 
        error_tables,
        optimal_allocation, 
        x_min_vec, 
        x_max_vec
    )

    # 3. Apply Precomputed Bounds (Tightening)
    for layer_idx, vars_in_layer in enumerate(all_layer_variables):
        lbs, ubs = precomputed_bounds[layer_idx]
        for neuron_idx, var in enumerate(vars_in_layer):
            var.lb = lbs[neuron_idx]
            var.ub = ubs[neuron_idx]
            
    model.update()
    
    # --- CRITICAL SETTINGS ---
    # Set a time limit per neuron (in seconds). 
    # Without this, it might run for days on 784 inputs.
    model.setParam('TimeLimit', time_limit) 
    model.setParam('MIPGap', output_layer_mip_gap)

    final_layer_vars = all_layer_variables[-1]
    min_bounds = np.zeros(len(final_layer_vars))
    max_bounds = np.zeros(len(final_layer_vars))

    for i, var in enumerate(final_layer_vars):
        # --- MINIMIZE ---
        model.setObjective(var, GRB.MINIMIZE)
        model.optimize()
        
        # Check Status
        if model.status == GRB.OPTIMAL:
            min_bounds[i] = model.ObjVal
        elif model.status == GRB.TIME_LIMIT:
            # If timed out, use the best proven bound found so far
            print(f"Warning: Neuron {i} MIN timed out. Using best bound.")
            min_bounds[i] = model.ObjBound 
        else:
            # Fallback to precomputed interval bounds if solver failed
            print(f"Error: Neuron {i} MIN failed (Status {model.status}).")
            min_bounds[i] = var.lb

        # --- MAXIMIZE ---
        model.setObjective(var, GRB.MAXIMIZE)
        model.optimize()
        
        if model.status == GRB.OPTIMAL:
            max_bounds[i] = model.ObjVal
        elif model.status == GRB.TIME_LIMIT:
            print(f"Warning: Neuron {i} MAX timed out. Using best bound.")
            max_bounds[i] = model.ObjBound
        else:
            print(f"Error: Neuron {i} MAX failed (Status {model.status}).")
            max_bounds[i] = var.ub

    return min_bounds, max_bounds

def verify_kan(my_kan, input_lb, input_ub):
    start_time = time.time()
    error_tables, segments_tables, lipschitz_constants = compute_dp_tables_lipschitz(my_kan, 30)
    weighted_error_tables = weight_dp_tables_lipschitz(my_kan, error_tables, segments_tables, lipschitz_constants)
    optimal_allocation, min_error = solve_best_segment_allocation(weighted_error_tables, 0.1)
    kan_shape = [my_kan.layers[0].input_dim]
    for layer in my_kan.layers:
        kan_shape.append(layer.output_dim)
    x_min_vec = np.full(kan_shape[0], input_lb)
    x_max_vec = np.full(kan_shape[0], input_ub) 
    output_layer_mip_gap = 0.1
    min_outputs, max_outputs = solve_kan_interval_milp(
        kan_shape,
        segments_tables, 
        error_tables,
        optimal_allocation,
        output_layer_mip_gap,
        x_min_vec, 
        x_max_vec
    )
    return min_outputs, max_outputs, time.time() - start_time

# MLP Verification Functions

In [2]:
def propagate_mlp_intervals(model, input_lb, input_ub):
    """
    Propagates interval bounds (Box domain) layer by layer to get 
    tight lb/ub for every intermediate neuron.
    """
    # Current bounds [Batch=1, Dim]
    current_lb = torch.tensor(input_lb, dtype=torch.float32).unsqueeze(0)
    current_ub = torch.tensor(input_ub, dtype=torch.float32).unsqueeze(0)
    
    layer_bounds = [] # Stores (lb, ub) for each layer output
    
    with torch.no_grad():
        for layer in model:
            if isinstance(layer, nn.Flatten):
                continue
            
            elif isinstance(layer, nn.Linear):
                weight = layer.weight
                bias = layer.bias
                
                # Separate positive and negative weights for bound calculations
                # W_pos * UB + W_neg * LB
                w_pos = torch.clamp(weight, min=0)
                w_neg = torch.clamp(weight, max=0)
                
                # Compute new Upper Bound
                # ub_new = W+ * ub_prev + W- * lb_prev + bias
                new_ub = (torch.matmul(w_pos, current_ub.t()) + 
                          torch.matmul(w_neg, current_lb.t())).t() + bias
                
                # Compute new Lower Bound
                # lb_new = W+ * lb_prev + W- * ub_prev + bias
                new_lb = (torch.matmul(w_pos, current_lb.t()) + 
                          torch.matmul(w_neg, current_ub.t())).t() + bias
                
                current_lb, current_ub = new_lb, new_ub
                # We store the pre-activation bounds (important for ReLU Big-M)
                layer_bounds.append((current_lb.numpy().flatten(), current_ub.numpy().flatten()))
                
            elif isinstance(layer, nn.ReLU):
                # Apply ReLU to bounds for the next iteration
                current_lb = torch.clamp(current_lb, min=0)
                current_ub = torch.clamp(current_ub, min=0)
                # We usually don't need to store post-activation bounds for the MILP 
                # formulation, as we use the pre-activation bounds to constrain the binary var.
                
    return layer_bounds

def milp_verify_mlp(model, input_lb, input_ub, timeout=300):
    # 1. PRE-COMPUTE TIGHT BOUNDS
    precomputed_bounds = propagate_mlp_intervals(model, input_lb, input_ub)
    
    m = gp.Model("mlp_verification")
    m.setParam("TimeLimit", timeout)
    m.setParam("OutputFlag", 0) 
    
    # 2. Define Input Variables
    input_vars = []
    for i in range(len(input_lb)):
        v = m.addVar(lb=input_lb[i], ub=input_ub[i], name=f"inp_{i}")
        input_vars.append(v)
    
    # Update to ensure input variables are registered
    m.update()
    
    current_vars = input_vars
    linear_layer_count = 0
    
    for layer in model:
        if isinstance(layer, nn.Flatten):
            continue
            
        elif isinstance(layer, nn.Linear):
            weight = layer.weight.detach().cpu().numpy()
            bias = layer.bias.detach().cpu().numpy()
            out_dim, in_dim = weight.shape
            
            # Retrieve pre-computed bounds for this linear output
            layer_lbs, layer_ubs = precomputed_bounds[linear_layer_count]
            
            next_vars = []
            for j in range(out_dim):
                lin_expr = gp.LinExpr(weight[j, :], current_vars) + bias[j]
                
                # TIGHTENING: Set variable bounds based on propagation
                lb_val = layer_lbs[j]
                ub_val = layer_ubs[j]
                
                out_v = m.addVar(lb=lb_val, ub=ub_val, name=f"lin{linear_layer_count}_{j}")
                m.addConstr(out_v == lin_expr)
                next_vars.append(out_v)
            
            current_vars = next_vars
            linear_layer_count += 1
            
            # --- FIX: FORCE UPDATE HERE ---
            # Gurobi needs this update to assign indices to the new variables
            # so that .LB and .UB can be accessed in the ReLU block.
            m.update() 

        elif isinstance(layer, nn.ReLU):
            # The input to this ReLU is the output of the previous Linear layer
            next_vars = []
            for j, pre_var in enumerate(current_vars):
                # Retrieve the bounds of the pre-activation variable
                l_bound = pre_var.LB
                u_bound = pre_var.UB
                
                # CASE 1: STABLE INACTIVE (Dead Neuron)
                if u_bound <= 0:
                    post_var = m.addVar(lb=0, ub=0, name=f"relu_dead_{j}")
                    next_vars.append(post_var)
                    
                # CASE 2: STABLE ACTIVE (Linear Neuron)
                elif l_bound >= 0:
                    post_var = m.addVar(lb=l_bound, ub=u_bound, name=f"relu_linear_{j}")
                    m.addConstr(post_var == pre_var)
                    next_vars.append(post_var)
                    
                # CASE 3: UNSTABLE (The only case needing Binary Variables)
                else:
                    post_var = m.addVar(lb=0.0, ub=u_bound, name=f"relu_{j}")
                    z = m.addVar(vtype=GRB.BINARY, name=f"relu_state_{j}")
                    
                    # TIGHT Big-M Constraints
                    # y >= x
                    m.addConstr(post_var >= pre_var)
                    
                    # y <= u * z
                    m.addConstr(post_var <= u_bound * z)
                    
                    # y <= x - l * (1 - z) 
                    m.addConstr(post_var <= pre_var - l_bound * (1 - z))
                    
                    next_vars.append(post_var)
                    
            current_vars = next_vars
            
            # Optional: Update after ReLU to finalize these vars too
            m.update()

    # Output Optimization
    output_vars = current_vars
    min_outputs, max_outputs = [], []
    
    for i in range(len(output_vars)):
        target_var = output_vars[i]
        
        m.setObjective(target_var, GRB.MINIMIZE)
        m.optimize()
        min_outputs.append(m.objVal if m.status == GRB.OPTIMAL else -np.inf)
        
        m.setObjective(target_var, GRB.MAXIMIZE)
        m.optimize()
        max_outputs.append(m.objVal if m.status == GRB.OPTIMAL else np.inf)

    return min_outputs, max_outputs

def verify_mlp(my_mlp, input_lb, input_ub):
    start_time = time.time()
    first_layer = my_mlp[0]
        
    input_dim = first_layer.weight.shape[1]
    x_min_vec = np.full(input_dim, input_lb) 
    x_max_vec = np.full(input_dim, input_ub)
    
    bounds_results = milp_verify_mlp(
        model=my_mlp, 
        input_lb=x_min_vec, 
        input_ub=x_max_vec,
        timeout=10000
    )
    return bounds_results, time.time() - start_time

# Gurobi Verification Comparison

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

from scipy.special import j0
import math
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

def train(model, get_data, steps=2000):
    opt = optim.AdamW(model.parameters(), lr=1e-2)
    loss_fn = nn.MSELoss()
    for i in range(steps):
        x, y = get_data()
        loss = loss_fn(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if i == steps - 1: 
            print(f"Step {i}, Loss: {loss.item():.6f}")

def get_optimal_mlp(input_dim, output_dim, target_params, depth):
    a = depth - 1
    b = input_dim + output_dim + depth
    c = output_dim - target_params
    if a == 0:
        width = -c / b
    else:
        delta = b**2 - 4*a*c
        if delta < 0:
            raise ValueError("Target parameters too low.")
        width = (-b + math.sqrt(delta)) / (2*a)
    
    width = int(round(width))
    width = max(width, 1)
    layers = []
    layers.append(nn.Linear(input_dim, width))
    layers.append(nn.ReLU())
    for _ in range(depth - 1):
        layers.append(nn.Linear(width, width))
        layers.append(nn.ReLU())
    layers.append(nn.Linear(width, output_dim))
    return nn.Sequential(*layers)

def get_optimal_fastkan(layers_hidden, target_params):
    var_per_grid = 0
    for in_dim, out_dim in zip(layers_hidden[:-1], layers_hidden[1:]):
        var_per_grid += in_dim * out_dim
    num_grids = int(round(target_params / var_per_grid))
    num_grids = max(1, num_grids)
    return FastKAN(layers_hidden, num_grids=num_grids, use_layernorm=False, use_base_update=False)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Initialize global loader once to avoid overhead in training loop
_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
_dataset = torchvision.datasets.MNIST(root="./data_mnist", train=True, download=True, transform=_transform)
_mnist_loader = DataLoader(_dataset, batch_size=1024, shuffle=True)
_mnist_iter = iter(_mnist_loader)

def data_6():
    global _mnist_iter
    try:
        x, y = next(_mnist_iter)
    except StopIteration:
        _mnist_iter = iter(_mnist_loader)
        x, y = next(_mnist_iter)
    x = x.view(x.size(0), -1)
    y = F.one_hot(y, num_classes=10).float()
    return x, y

kan_6 = get_optimal_fastkan([784, 13, 10], 30000)
mlp_6 = get_optimal_mlp(784, 10, 30000, 3)
print(f"FastKAN Params: {count_params(kan_6)} | MLP Params: {count_params(mlp_6)}")

print("Training FastKAN...")
train(kan_6, data_6, steps=4)
print("Training MLP...")
train(mlp_6, data_6, steps=4)

print("Verifying FastKAN...")
# Only printing Neuron 0 for brevity
k_min, k_max, k_time = verify_kan(kan_6, -0.005, 0.005)
print(f"Result (Neuron 0): [{k_min[0]:.4f}, {k_max[0]:.4f}] in {k_time:.6f}s")

print("Verifying MLP...")
(m_min, m_max), m_time = verify_mlp(mlp_6, -0.005, 0.005)
print(f"Result (Neuron 0): [{m_min[0]:.4f}, {m_max[0]:.4f}] in {m_time:.6f}s")

FastKAN Params: 30966 | MLP Params: 30355
Training FastKAN...
Step 3, Loss: 0.100000
Training MLP...
Step 3, Loss: 0.112525
Verifying FastKAN...
Analyzing Layer 0: 784 inputs -> 13 outputs
Analyzing Layer 1: 13 inputs -> 10 outputs
Set parameter Username
Set parameter LicenseID to value 2620375
Academic license - for non-commercial use only - expires 2026-02-10
Optimal allocation found with 242526.0 segments.
Error: Neuron 0 MIN failed (Status 3).
Error: Neuron 0 MAX failed (Status 3).
Error: Neuron 1 MIN failed (Status 3).
Error: Neuron 1 MAX failed (Status 3).
Error: Neuron 2 MIN failed (Status 3).
Error: Neuron 2 MAX failed (Status 3).
Error: Neuron 3 MIN failed (Status 3).
Error: Neuron 3 MAX failed (Status 3).
Error: Neuron 4 MIN failed (Status 3).
Error: Neuron 4 MAX failed (Status 3).
Error: Neuron 5 MIN failed (Status 3).
Error: Neuron 5 MAX failed (Status 3).
Error: Neuron 6 MIN failed (Status 3).
Error: Neuron 6 MAX failed (Status 3).
Error: Neuron 7 MIN failed (Status 3).
Er